# 正样本对获取方式详解

分析 MultiSimilarityMiner 如何从标签矩阵中获取正样本对


In [14]:
"""
详细展示正样本对的获取过程：从标签矩阵到最终的正样本对
"""
import torch
import torch.nn.functional as F

# 使用给定的数据
descriptors = torch.tensor([[ 0.5484, -1.8406,  0.3203, -0.4526],
        [-0.4881,  0.4921, -0.2268,  0.4315],
        [ 0.7441, -1.3973,  0.4317, -1.0654],
        [-0.7276,  0.2828,  1.8962,  0.4187],
        [ 1.4946, -1.1203, -1.5092,  0.0088],
        [-1.3241,  1.1952, -0.4905,  0.4526],
        [-0.6548, -0.2782, -0.6012,  0.1352],
        [ 0.2919,  0.8895,  0.1728, -0.4657]])

labels = torch.tensor([0, 0, 1, 1, 2, 2, 3, 3])

print("=" * 100)
print("正样本对获取方式详解")
print("=" * 100)

print(f"\n【步骤1】输入数据")
print(f"  descriptors.shape = {descriptors.shape}")
print(f"  labels = {labels.tolist()}")
print(f"  标签分布:")
for label in torch.unique(labels):
    indices = torch.where(labels == label)[0]
    print(f"    标签{label.item()}: 样本 {indices.tolist()}")

# 计算余弦相似度矩阵
def cosine_similarity_matrix(x):
    x_norm = x / (x.norm(dim=1, keepdim=True) + 1e-8)
    sim_matrix = torch.mm(x_norm, x_norm.t())
    return sim_matrix

cosine_sim = cosine_similarity_matrix(descriptors)

# 创建标签匹配矩阵
labels_expanded = labels.unsqueeze(1)  # [8, 1]
same_label_mask = (labels_expanded == labels_expanded.t()).float()  # [8, 8]

print(f"\n【步骤2】创建标签匹配矩阵")
print(f"  标签匹配矩阵 [8, 8] (1=相同标签, 0=不同标签):")
print(f"  " + " ".join([f"{i:>4}" for i in range(8)]))
for i in range(8):
    row_str = f"  {i} " + " ".join([f"{int(same_label_mask[i, j].item()):>4}" 
                                   for j in range(8)])
    print(row_str)

print(f"\n【步骤3】正样本对获取过程（关键步骤）")
print("-" * 100)
print("""
正样本对的获取方式：
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

1. 从标签矩阵中根据数字为1的索引获取候选正样本
   - 对于每个锚点i，在same_label_mask[i, :]中找到所有值为1的位置
   - 这些位置j对应的样本就是与锚点i标签相同的候选正样本
   - 排除自己（对角线元素）

2. 根据相似度筛选困难正样本对
   - 计算锚点i与所有候选正样本的相似度
   - 找到最相似的正样本相似度 max_pos_sim
   - 筛选出相似度 < (max_pos_sim - epsilon) 的困难正样本
   - epsilon = 0.1 控制困难程度

3. 不是随机抽取！
   - 正样本对是从标签矩阵中数字为1的位置获取的
   - 但还需要根据相似度进一步筛选困难样本对
   - 只选择那些相似度较低的正样本对（困难样本）

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
""")

print(f"\n【步骤4】详细演示：以样本0为例")
print("-" * 100)
anchor_idx = 0
anchor_label = labels[anchor_idx].item()

print(f"  锚点: 样本{anchor_idx} (标签={anchor_label})")

# 方法1：从标签矩阵中获取（same_label_mask中值为1的位置）
print(f"\n  方法1：从标签矩阵中获取候选正样本")
print(f"    查看 same_label_mask[{anchor_idx}, :] = {same_label_mask[anchor_idx].int().tolist()}")
print(f"    值为1的位置（相同标签）: ", end="")
same_label_positions = torch.where(same_label_mask[anchor_idx] == 1)[0]
print(f"{same_label_positions.tolist()}")

# 排除自己
positive_candidates_from_mask = same_label_positions[same_label_positions != anchor_idx]
print(f"    排除自己后: {positive_candidates_from_mask.tolist()}")

# 方法2：直接从labels中获取（验证两种方法结果一致）
print(f"\n  方法2：直接从labels中获取（验证）")
positive_candidates_from_labels = torch.where(labels == anchor_label)[0]
positive_candidates_from_labels = positive_candidates_from_labels[positive_candidates_from_labels != anchor_idx]
print(f"    标签为{anchor_label}的样本: {positive_candidates_from_labels.tolist()}")

print(f"\n  验证: 两种方法结果 {'一致 ✓' if torch.equal(positive_candidates_from_mask, positive_candidates_from_labels) else '不一致 ✗'}")

# 计算相似度并筛选困难正样本
if len(positive_candidates_from_mask) > 0:
    pos_similarities = cosine_sim[anchor_idx, positive_candidates_from_mask]
    print(f"\n  步骤5：计算相似度并筛选困难正样本")
    print(f"    候选正样本与锚点的相似度:")
    for i, pos_idx in enumerate(positive_candidates_from_mask):
        print(f"      样本{pos_idx}: {pos_similarities[i].item():.4f}")
    
    max_pos_sim = pos_similarities.max()
    threshold_pos = max_pos_sim - 0.1  # epsilon = 0.1
    
    print(f"\n    最相似的正样本相似度: {max_pos_sim.item():.4f}")
    print(f"    困难正样本阈值: {threshold_pos.item():.4f} (max_pos_sim - epsilon)")
    
    hard_positives = positive_candidates_from_mask[pos_similarities < threshold_pos]
    print(f"    困难正样本: {hard_positives.tolist()} (相似度 < {threshold_pos.item():.4f})")
    
    if len(hard_positives) > 0:
        print(f"\n    最终挖掘出的正样本对:")
        for pos_idx in hard_positives:
            sim_val = cosine_sim[anchor_idx, pos_idx].item()
            print(f"      (锚点{anchor_idx}, 正样本{pos_idx.item()}): 相似度={sim_val:.4f}")
    else:
        print(f"\n    没有挖掘出困难正样本对（所有正样本相似度都较高）")

print(f"\n【步骤6】使用官方Miner验证")
print("-" * 100)
from pytorch_metric_learning import miners
from pytorch_metric_learning.distances import CosineSimilarity

miner = miners.MultiSimilarityMiner(epsilon=0.1, distance=CosineSimilarity())
miner_outputs = miner(descriptors, labels)
tensor_a, tensor_b, tensor_c, tensor_d = miner_outputs

print(f"  官方Miner挖掘结果:")
print(f"    正样本对数量: {len(tensor_a)}")
print(f"    前5个正样本对: ", end="")
for i in range(min(5, len(tensor_a))):
    print(f"({tensor_a[i].item()}, {tensor_b[i].item()})", end=" ")
print()

# 验证样本0作为锚点的正样本对
anchor_0_pos_pairs = [(tensor_a[i].item(), tensor_b[i].item()) 
                       for i in range(len(tensor_a)) if tensor_a[i].item() == 0]
print(f"\n  样本0作为锚点的正样本对: {anchor_0_pos_pairs}")

print("\n" + "=" * 100)
print("总结")
print("=" * 100)
print("""
正样本对的获取方式：

1. ✓ 从标签矩阵中根据数字为1的索引获取候选正样本
   - 使用 same_label_mask[i, j] == 1 找到所有标签相同的样本
   - 这是第一步筛选：确保标签相同

2. ✓ 根据相似度进一步筛选困难正样本对
   - 计算锚点与所有候选正样本的相似度
   - 筛选出相似度 < (max_pos_sim - epsilon) 的困难样本
   - 这是第二步筛选：确保是困难样本

3. ✗ 不是随机抽取
   - 整个过程都是确定性的
   - 基于标签匹配和相似度阈值
   - 没有任何随机性

关键点：
- 标签矩阵（same_label_mask）用于第一步：找到所有相同标签的样本
- 相似度矩阵用于第二步：筛选出困难正样本对
- 两步筛选确保只选择标签相同且相似度较低的正样本对
""")


正样本对获取方式详解

【步骤1】输入数据
  descriptors.shape = torch.Size([8, 4])
  labels = [0, 0, 1, 1, 2, 2, 3, 3]
  标签分布:
    标签0: 样本 [0, 1]
    标签1: 样本 [2, 3]
    标签2: 样本 [4, 5]
    标签3: 样本 [6, 7]

【步骤2】创建标签匹配矩阵
  标签匹配矩阵 [8, 8] (1=相同标签, 0=不同标签):
     0    1    2    3    4    5    6    7
  0    1    1    0    0    0    0    0    0
  1    1    1    0    0    0    0    0    0
  2    0    0    1    1    0    0    0    0
  3    0    0    1    1    0    0    0    0
  4    0    0    0    0    1    1    0    0
  5    0    0    0    0    1    1    0    0
  6    0    0    0    0    0    0    1    1
  7    0    0    0    0    0    0    1    1

【步骤3】正样本对获取过程（关键步骤）
----------------------------------------------------------------------------------------------------

正样本对的获取方式：
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

1. 从标签矩阵中根据数字为1的索引获取候选正样本
   - 对于每个锚点i，在same_label_mask[i, :]中找到所有值为1的位置
   - 这些位置j对应的样本就是与锚点i标签相同的候选正样本
   - 排除自己（对角线元素）

2. 根据相似度筛选困难正样本对
   - 计算锚点i与所有候选正样本

# 为什么负样本对的标签看起来"随机"？

分析负样本对标签的获取逻辑


In [15]:
"""
详细分析：为什么负样本对的标签看起来"随机"
"""
import torch
import torch.nn.functional as F

# 使用给定的数据
descriptors = torch.tensor([[ 0.5484, -1.8406,  0.3203, -0.4526],
        [-0.4881,  0.4921, -0.2268,  0.4315],
        [ 0.7441, -1.3973,  0.4317, -1.0654],
        [-0.7276,  0.2828,  1.8962,  0.4187],
        [ 1.4946, -1.1203, -1.5092,  0.0088],
        [-1.3241,  1.1952, -0.4905,  0.4526],
        [-0.6548, -0.2782, -0.6012,  0.1352],
        [ 0.2919,  0.8895,  0.1728, -0.4657]])

labels = torch.tensor([0, 0, 1, 1, 2, 2, 3, 3])

print("=" * 100)
print("为什么负样本对的标签看起来'随机'？")
print("=" * 100)

# 计算余弦相似度矩阵
def cosine_similarity_matrix(x):
    x_norm = x / (x.norm(dim=1, keepdim=True) + 1e-8)
    sim_matrix = torch.mm(x_norm, x_norm.t())
    return sim_matrix

cosine_sim = cosine_similarity_matrix(descriptors)

# 创建标签匹配矩阵
labels_expanded = labels.unsqueeze(1)
same_label_mask = (labels_expanded == labels_expanded.t()).float()

print(f"\n【关键理解】负样本对的标签不是随机的！")
print("-" * 100)
print("""
负样本对的标签看起来"随机"的原因：
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

1. 负样本对的定义：标签必须不同
   - 负样本对 = (负样本锚点, 负样本)
   - 要求：labels[负样本锚点] != labels[负样本]
   - 这是硬性要求，不是随机的

2. 负样本对的筛选：基于相似度
   - 从所有标签不同的样本对中，筛选出相似度较高的困难负样本对
   - 相似度 > (min_neg_sim + epsilon)
   - 这是基于特征相似度的确定性筛选

3. 为什么看起来"随机"？
   - 因为负样本锚点可能来自不同标签（0, 1, 2, 3）
   - 负样本也可能来自不同标签（只要与锚点标签不同）
   - 标签的组合看起来多样，但都是确定性的

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
""")

print(f"\n【步骤1】分析标签分布")
print("-" * 100)
print(f"  labels = {labels.tolist()}")
print(f"  标签分布:")
for label in torch.unique(labels):
    indices = torch.where(labels == label)[0]
    print(f"    标签{label.item()}: 样本 {indices.tolist()}")

print(f"\n【步骤2】负样本对获取过程（以样本0为例）")
print("-" * 100)
neg_anchor_idx = 0
neg_anchor_label = labels[neg_anchor_idx].item()

print(f"  负样本锚点: 样本{neg_anchor_idx} (标签={neg_anchor_label})")

# 从标签矩阵中获取所有标签不同的样本
print(f"\n  步骤1：从标签矩阵中获取候选负样本")
print(f"    查看 same_label_mask[{neg_anchor_idx}, :] = {same_label_mask[neg_anchor_idx].int().tolist()}")
print(f"    值为0的位置（不同标签）: ", end="")
different_label_positions = torch.where(same_label_mask[neg_anchor_idx] == 0)[0]
print(f"{different_label_positions.tolist()}")

print(f"\n    这些候选负样本的标签:")
for neg_idx in different_label_positions:
    neg_label = labels[neg_idx].item()
    print(f"      样本{neg_idx}: 标签={neg_label} (与锚点标签{neg_anchor_label}不同 ✓)")

# 计算相似度并筛选困难负样本
if len(different_label_positions) > 0:
    neg_similarities = cosine_sim[neg_anchor_idx, different_label_positions]
    print(f"\n  步骤2：计算相似度并筛选困难负样本")
    print(f"    候选负样本与锚点的相似度:")
    for i, neg_idx in enumerate(different_label_positions):
        sim_val = neg_similarities[i].item()
        neg_label = labels[neg_idx].item()
        print(f"      样本{neg_idx}(标签={neg_label}): {sim_val:.4f}")
    
    min_neg_sim = neg_similarities.min()
    threshold_neg = min_neg_sim + 0.1  # epsilon = 0.1
    
    print(f"\n    最不相似的负样本相似度: {min_neg_sim.item():.4f}")
    print(f"    困难负样本阈值: {threshold_neg.item():.4f} (min_neg_sim + epsilon)")
    
    hard_negatives = different_label_positions[neg_similarities > threshold_neg]
    print(f"    困难负样本: {hard_negatives.tolist()} (相似度 > {threshold_neg.item():.4f})")
    
    if len(hard_negatives) > 0:
        print(f"\n    最终挖掘出的负样本对及其标签:")
        for neg_idx in hard_negatives:
            sim_val = cosine_sim[neg_anchor_idx, neg_idx].item()
            neg_label = labels[neg_idx].item()
            print(f"      (负锚点{neg_anchor_idx}, 负样本{neg_idx}): 标签={neg_anchor_label} vs {neg_label}, 相似度={sim_val:.4f}")

print(f"\n【步骤3】使用官方Miner验证并分析标签分布")
print("-" * 100)
from pytorch_metric_learning import miners
from pytorch_metric_learning.distances import CosineSimilarity

miner = miners.MultiSimilarityMiner(epsilon=0.1, distance=CosineSimilarity())
miner_outputs = miner(descriptors, labels)
tensor_a, tensor_b, tensor_c, tensor_d = miner_outputs

print(f"  负样本对总数: {len(tensor_c)}")
print(f"  负样本锚点分布: {dict(zip(*torch.unique(tensor_c, return_counts=True)))}")

# 分析负样本对的标签组合
print(f"\n  负样本对的标签组合分析:")
neg_pair_label_combinations = {}
for i in range(len(tensor_c)):
    neg_anchor_idx = tensor_c[i].item()
    neg_idx = tensor_d[i].item()
    neg_anchor_label = labels[neg_anchor_idx].item()
    neg_label = labels[neg_idx].item()
    
    # 验证标签不同
    assert neg_anchor_label != neg_label, f"错误：负样本对标签相同！"
    
    key = f"{neg_anchor_label} vs {neg_label}"
    if key not in neg_pair_label_combinations:
        neg_pair_label_combinations[key] = []
    neg_pair_label_combinations[key].append((neg_anchor_idx, neg_idx))

print(f"    标签组合统计:")
for key, pairs in sorted(neg_pair_label_combinations.items()):
    print(f"      {key}: {len(pairs)}个负样本对")
    print(f"        示例: {pairs[:3]}")

print(f"\n【步骤4】为什么看起来'随机'？")
print("-" * 100)
print("""
原因分析：
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

1. 负样本锚点来自不同标签
   - 每个样本都可以作为负样本锚点
   - 标签0, 1, 2, 3的样本都可能成为负样本锚点
   - 所以负样本锚点的标签看起来多样

2. 负样本也来自不同标签（只要与锚点不同）
   - 对于锚点标签0，负样本可以是标签1, 2, 3的任意样本
   - 对于锚点标签1，负样本可以是标签0, 2, 3的任意样本
   - 所以负样本的标签也看起来多样

3. 标签组合的多样性
   - 可能的组合：0 vs 1, 0 vs 2, 0 vs 3, 1 vs 0, 1 vs 2, 1 vs 3, ...
   - 组合数量 = 标签数 × (标签数 - 1) = 4 × 3 = 12种可能
   - 但实际挖掘的数量取决于困难样本的分布

4. 但整个过程是确定性的！
   - 不是随机抽取
   - 基于标签匹配（必须不同）和相似度筛选（困难样本）
   - 相同的输入总是产生相同的输出

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
""")

print(f"\n【步骤5】验证确定性（运行两次应该得到相同结果）")
print("-" * 100)
miner_outputs_1 = miner(descriptors, labels)
miner_outputs_2 = miner(descriptors, labels)

tensor_c1, tensor_d1 = miner_outputs_1[2], miner_outputs_1[3]
tensor_c2, tensor_d2 = miner_outputs_2[2], miner_outputs_2[3]

# 转换为列表进行比较
pairs1 = [(tensor_c1[i].item(), tensor_d1[i].item()) for i in range(len(tensor_c1))]
pairs2 = [(tensor_c2[i].item(), tensor_d2[i].item()) for i in range(len(tensor_c2))]

print(f"  第一次运行: {len(pairs1)}个负样本对")
print(f"  第二次运行: {len(pairs2)}个负样本对")
print(f"  结果是否相同: {'是 ✓ (确定性)' if pairs1 == pairs2 else '否 ✗ (随机性)'}")

print("\n" + "=" * 100)
print("总结")
print("=" * 100)
print("""
负样本对的标签不是随机的！

1. ✓ 标签必须不同（硬性要求）
   - 负样本对 = (负样本锚点, 负样本)
   - 要求：labels[负样本锚点] != labels[负样本]

2. ✓ 基于相似度筛选（确定性）
   - 从所有标签不同的样本对中筛选
   - 选择相似度较高的困难负样本对
   - 相似度 > (min_neg_sim + epsilon)

3. ✓ 看起来"随机"的原因
   - 负样本锚点可能来自不同标签
   - 负样本也可能来自不同标签（只要与锚点不同）
   - 标签组合多样，但都是确定性的

4. ✗ 不是随机抽取
   - 整个过程是确定性的
   - 相同的输入总是产生相同的输出
   - 基于标签匹配和相似度阈值

关键点：
- 标签的多样性来自于：不同标签的样本都可以成为负样本对
- 但选择哪些样本对是确定性的，基于相似度筛选
- 目的是找到困难负样本对（标签不同但相似度高），需要推远
""")


为什么负样本对的标签看起来'随机'？

【关键理解】负样本对的标签不是随机的！
----------------------------------------------------------------------------------------------------

负样本对的标签看起来"随机"的原因：
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

1. 负样本对的定义：标签必须不同
   - 负样本对 = (负样本锚点, 负样本)
   - 要求：labels[负样本锚点] != labels[负样本]
   - 这是硬性要求，不是随机的

2. 负样本对的筛选：基于相似度
   - 从所有标签不同的样本对中，筛选出相似度较高的困难负样本对
   - 相似度 > (min_neg_sim + epsilon)
   - 这是基于特征相似度的确定性筛选

3. 为什么看起来"随机"？
   - 因为负样本锚点可能来自不同标签（0, 1, 2, 3）
   - 负样本也可能来自不同标签（只要与锚点标签不同）
   - 标签的组合看起来多样，但都是确定性的

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━


【步骤1】分析标签分布
----------------------------------------------------------------------------------------------------
  labels = [0, 0, 1, 1, 2, 2, 3, 3]
  标签分布:
    标签0: 样本 [0, 1]
    标签1: 样本 [2, 3]
    标签2: 样本 [4, 5]
    标签3: 样本 [6, 7]

【步骤2】负样本对获取过程（以样本0为例）
----------------------------------------------------------------------------------------------------
  负样本锚点:

# MultiSimilarityMiner 核心代码和完整过程

展示 MultiSimilarityMiner 的完整实现代码和逐步执行过程


In [4]:
"""
MultiSimilarityMiner 核心代码完整实现和逐步执行过程
"""
import torch
import torch.nn.functional as F

# 使用给定的数据
descriptors = torch.tensor([[ 0.5484, -1.8406,  0.3203, -0.4526],
        [-0.4881,  0.4921, -0.2268,  0.4315],
        [ 0.7441, -1.3973,  0.4317, -1.0654],
        [-0.7276,  0.2828,  1.8962,  0.4187],
        [ 1.4946, -1.1203, -1.5092,  0.0088],
        [-1.3241,  1.1952, -0.4905,  0.4526],
        [-0.6548, -0.2782, -0.6012,  0.1352],
        [ 0.2919,  0.8895,  0.1728, -0.4657]])

labels = torch.tensor([0, 0, 1, 1, 2, 2, 3, 3])

print("=" * 100)
print("MultiSimilarityMiner 核心代码和完整过程")
print("=" * 100)

# ==================== 核心代码实现 ====================
def multi_similarity_miner_core(embeddings, labels, epsilon=0.1):
    """
    MultiSimilarityMiner 核心实现
    
    参数:
        embeddings: [N, D] 特征向量
        labels: [N] 标签
        epsilon: 困难样本阈值
    
    返回:
        positive_pairs: [(anchor_idx, positive_idx), ...]
        negative_pairs: [(negative_anchor_idx, negative_idx), ...]
    """
    N = embeddings.size(0)
    
    print(f"\n【核心代码步骤1】计算相似度矩阵")
    print(f"  输入: embeddings.shape = {embeddings.shape}")
    
    # 步骤1: 计算余弦相似度矩阵
    embeddings_norm = F.normalize(embeddings, p=2, dim=1)  # [N, D] 归一化
    sim_matrix = torch.mm(embeddings_norm, embeddings_norm.t())  # [N, N]
    
    print(f"  归一化特征: embeddings_norm.shape = {embeddings_norm.shape}")
    print(f"  相似度矩阵: sim_matrix.shape = {sim_matrix.shape}")
    print(f"  相似度范围: [{sim_matrix.min().item():.4f}, {sim_matrix.max().item():.4f}]")
    
    print(f"\n【核心代码步骤2】创建标签匹配矩阵")
    # 步骤2: 创建标签匹配矩阵
    labels_expanded = labels.unsqueeze(1)  # [N, 1]
    same_label_mask = (labels_expanded == labels_expanded.t()).float()  # [N, N]
    # 1表示相同标签，0表示不同标签
    
    print(f"  labels_expanded.shape = {labels_expanded.shape}")
    print(f"  same_label_mask.shape = {same_label_mask.shape}")
    print(f"  相同标签对数量: {(same_label_mask.sum().item() - N) / 2:.0f} (去掉对角线，除以2去重)")
    print(f"  不同标签对数量: {((1 - same_label_mask).sum().item() - N) / 2:.0f} (去掉对角线，除以2去重)")
    
    print(f"\n【核心代码步骤3】挖掘困难正样本对")
    # 步骤3: 挖掘困难正样本对
    positive_pairs = []
    
    for anchor_idx in range(N):
        # 找到所有与锚点标签相同的样本（正样本候选）
        positive_candidates = torch.where(same_label_mask[anchor_idx] == 1)[0]
        # 排除自己
        positive_candidates = positive_candidates[positive_candidates != anchor_idx]
        print(f"  锚点{anchor_idx}的正样本候选: {positive_candidates.tolist()}")
        if len(positive_candidates) > 0:
            # 获取锚点与所有正样本候选的相似度
            pos_similarities = sim_matrix[anchor_idx, positive_candidates]
            
            # 找到最相似的正样本（最容易的正样本）
            max_pos_sim = pos_similarities.max()
            
            # 挖掘困难正样本: 相似度 < (最相似的正样本相似度 - epsilon)
            threshold_pos = max_pos_sim - epsilon
            
            # 找到困难正样本
            hard_positives = positive_candidates[pos_similarities < threshold_pos]
            
            # 添加到正样本对列表
            for pos_idx in hard_positives:
                positive_pairs.append((anchor_idx, pos_idx.item()))
    
    print(f"  正样本对数量: {len(positive_pairs)}")
    
    print(f"\n【核心代码步骤4】挖掘困难负样本对")
    # 步骤4: 挖掘困难负样本对
    negative_pairs = []
    
    for neg_anchor_idx in range(N):
        # 找到所有与负样本锚点标签不同的样本（负样本候选）
        negative_candidates = torch.where(same_label_mask[neg_anchor_idx] == 0)[0]
        print(f"  负样本锚点{neg_anchor_idx}的负样本候选: {negative_candidates.tolist()}")
        if len(negative_candidates) > 0:
            # 获取负样本锚点与所有负样本候选的相似度
            neg_similarities = sim_matrix[neg_anchor_idx, negative_candidates]
            
            # 找到最不相似的负样本（最容易的负样本）
            min_neg_sim = neg_similarities.min()
            
            # 挖掘困难负样本: 相似度 > (最不相似的负样本相似度 + epsilon)
            threshold_neg = min_neg_sim + epsilon
            
            # 找到困难负样本
            hard_negatives = negative_candidates[neg_similarities > threshold_neg]
            
            # 添加到负样本对列表
            for neg_idx in hard_negatives:
                negative_pairs.append((neg_anchor_idx, neg_idx.item()))
    
    print(f"  负样本对数量: {len(negative_pairs)}")
    
    return positive_pairs, negative_pairs, sim_matrix, same_label_mask

# ==================== 执行核心代码 ====================
print(f"\n输入数据:")
print(f"  descriptors.shape = {descriptors.shape}")
print(f"  labels = {labels.tolist()}")
print(f"  epsilon = 0.1")

positive_pairs, negative_pairs, sim_matrix, same_label_mask = multi_similarity_miner_core(
    descriptors, labels, epsilon=0.1
)

print(f"\n【结果】")
print(f"  正样本对: {len(positive_pairs)}个")
print(f"  负样本对: {len(negative_pairs)}个")
print(f"  前5个正样本对: {positive_pairs[:10]}")
print(f"  前5个负样本对: {negative_pairs[:10]}")

# ==================== 详细展示每个步骤 ====================
print(f"\n" + "=" * 100)
print("详细展示每个步骤的执行过程")
print("=" * 100)

print(f"\n【步骤1详细】相似度矩阵计算")
print("-" * 100)
print(f"  输入特征向量（前3个样本）:")
for i in range(3):
    print(f"    样本{i}: {descriptors[i].tolist()}")
    norm_val = descriptors[i].norm().item()
    print(f"      范数: {norm_val:.4f}")

embeddings_norm = F.normalize(descriptors, p=2, dim=1)
print(f"\n  归一化后的特征向量（前3个样本）:")
for i in range(3):
    print(f"    样本{i}: {embeddings_norm[i].tolist()}")
    norm_val = embeddings_norm[i].norm().item()
    print(f"      范数: {norm_val:.4f} (归一化后)")

print(f"\n  相似度矩阵（前8x8）:")
print(f"  " + " ".join([f"{i:>8}" for i in range(8)]))
for i in range(8):
    row_str = f"  {i} " + " ".join([f"{sim_matrix[i, j].item():8.4f}" for j in range(8)])
    print(row_str)

print(f"\n【步骤2详细】标签匹配矩阵")
print("-" * 100)
print(f"  标签: {labels.tolist()}")
print(f"  标签匹配矩阵 [8, 8]:")
print(f"  " + " ".join([f"{i:>4}" for i in range(8)]))
for i in range(8):
    row_str = f"  {i} " + " ".join([f"{int(same_label_mask[i, j].item()):>4}" 
                                   for j in range(8)])
    print(row_str)

print(f"\n【步骤3详细】正样本对挖掘过程（以样本0为例）")
print("-" * 100)
anchor_idx = 0
anchor_label = labels[anchor_idx].item()

print(f"  锚点: 样本{anchor_idx} (标签={anchor_label})")

# 找到所有与锚点标签相同的样本
positive_candidates = torch.where(same_label_mask[anchor_idx] == 1)[0]
positive_candidates = positive_candidates[positive_candidates != anchor_idx]

print(f"  候选正样本: {positive_candidates.tolist()}")

if len(positive_candidates) > 0:
    pos_similarities = sim_matrix[anchor_idx, positive_candidates]
    print(f"  相似度:")
    for i, pos_idx in enumerate(positive_candidates):
        print(f"    样本{pos_idx}: {pos_similarities[i].item():.4f}")
    
    max_pos_sim = pos_similarities.max()
    threshold_pos = max_pos_sim - 0.1
    
    print(f"\n  最相似的正样本相似度: {max_pos_sim.item():.4f}")
    print(f"  困难正样本阈值: {threshold_pos.item():.4f} = {max_pos_sim.item():.4f} - 0.1")
    
    hard_positives = positive_candidates[pos_similarities < threshold_pos]
    print(f"  困难正样本: {hard_positives.tolist()}")
    
    if len(hard_positives) > 0:
        print(f"  挖掘出的正样本对:")
        for pos_idx in hard_positives:
            print(f"    ({anchor_idx}, {pos_idx.item()})")
    else:
        print(f"  没有挖掘出困难正样本对")

print(f"\n【步骤4详细】负样本对挖掘过程（以样本0为例）")
print("-" * 100)
neg_anchor_idx = 0
neg_anchor_label = labels[neg_anchor_idx].item()

print(f"  负样本锚点: 样本{neg_anchor_idx} (标签={neg_anchor_label})")

# 找到所有与负样本锚点标签不同的样本
negative_candidates = torch.where(same_label_mask[neg_anchor_idx] == 0)[0]

print(f"  候选负样本: {negative_candidates.tolist()}")

if len(negative_candidates) > 0:
    neg_similarities = sim_matrix[neg_anchor_idx, negative_candidates]
    print(f"  相似度:")
    for i, neg_idx in enumerate(negative_candidates):
        neg_label = labels[neg_idx].item()
        print(f"    样本{neg_idx}(标签={neg_label}): {neg_similarities[i].item():.4f}")
    
    min_neg_sim = neg_similarities.min()
    threshold_neg = min_neg_sim + 0.1
    
    print(f"\n  最不相似的负样本相似度: {min_neg_sim.item():.4f}")
    print(f"  困难负样本阈值: {threshold_neg.item():.4f} = {min_neg_sim.item():.4f} + 0.1")
    
    hard_negatives = negative_candidates[neg_similarities > threshold_neg]
    print(f"  困难负样本: {hard_negatives.tolist()}")
    
    if len(hard_negatives) > 0:
        print(f"  挖掘出的负样本对:")
        for neg_idx in hard_negatives:
            neg_label = labels[neg_idx].item()
            print(f"    ({neg_anchor_idx}, {neg_idx.item()}) - 标签: {neg_anchor_label} vs {neg_label}")

# ==================== 与官方实现对比 ====================
print(f"\n" + "=" * 100)
print("与官方 MultiSimilarityMiner 对比")
print("=" * 100)

from pytorch_metric_learning import miners
from pytorch_metric_learning.distances import CosineSimilarity

official_miner = miners.MultiSimilarityMiner(epsilon=0.1, distance=CosineSimilarity())
official_outputs = official_miner(descriptors, labels)
tensor_a, tensor_b, tensor_c, tensor_d = official_outputs

official_pos_pairs = [(tensor_a[i].item(), tensor_b[i].item()) 
                      for i in range(len(tensor_a))]
official_neg_pairs = [(tensor_c[i].item(), tensor_d[i].item()) 
                      for i in range(len(tensor_c))]

print(f"\n官方实现结果:")
print(f"  正样本对: {len(official_pos_pairs)}个")
print(f"  负样本对: {len(official_neg_pairs)}个")

print(f"\n简化实现结果:")
print(f"  正样本对: {len(positive_pairs)}个")
print(f"  负样本对: {len(negative_pairs)}个")

# 比较结果（注意顺序可能不同）
pos_set1 = set(positive_pairs)
pos_set2 = set(official_pos_pairs)
neg_set1 = set(negative_pairs)
neg_set2 = set(official_neg_pairs)

print(f"\n结果比较:")
print(f"  正样本对集合是否相同: {pos_set1 == pos_set2}")
print(f"  负样本对集合是否相同: {neg_set1 == neg_set2}")

if pos_set1 != pos_set2:
    print(f"  正样本对差异:")
    print(f"    简化实现有但官方没有: {pos_set1 - pos_set2}")
    print(f"    官方有但简化实现没有: {pos_set2 - pos_set1}")

if neg_set1 != neg_set2:
    print(f"  负样本对差异:")
    print(f"    简化实现有但官方没有: {neg_set1 - neg_set2}")
    print(f"    官方有但简化实现没有: {neg_set2 - neg_set1}")

print("\n" + "=" * 100)
print("核心代码总结")
print("=" * 100)
print("""
MultiSimilarityMiner 核心代码流程：

1. 计算相似度矩阵 [N, N]
   - 归一化特征向量
   - 计算余弦相似度矩阵

2. 创建标签匹配矩阵 [N, N]
   - same_label_mask[i, j] = 1 如果 labels[i] == labels[j]
   - same_label_mask[i, j] = 0 如果 labels[i] != labels[j]

3. 挖掘困难正样本对
   - 对于每个锚点i，找到所有标签相同的样本j
   - 计算相似度，找到最相似的正样本相似度 max_pos_sim
   - 筛选困难正样本: 相似度 < (max_pos_sim - epsilon)

4. 挖掘困难负样本对
   - 对于每个负样本锚点i，找到所有标签不同的样本j
   - 计算相似度，找到最不相似的负样本相似度 min_neg_sim
   - 筛选困难负样本: 相似度 > (min_neg_sim + epsilon)

关键参数:
- epsilon: 控制困难程度阈值（默认0.1）
- 正样本对: 标签相同 + 相似度低 → 需要拉近
- 负样本对: 标签不同 + 相似度高 → 需要推远
""")


MultiSimilarityMiner 核心代码和完整过程

输入数据:
  descriptors.shape = torch.Size([8, 4])
  labels = [0, 0, 1, 1, 2, 2, 3, 3]
  epsilon = 0.1

【核心代码步骤1】计算相似度矩阵
  输入: embeddings.shape = torch.Size([8, 4])
  归一化特征: embeddings_norm.shape = torch.Size([8, 4])
  相似度矩阵: sim_matrix.shape = torch.Size([8, 8])
  相似度范围: [-0.9702, 1.0000]

【核心代码步骤2】创建标签匹配矩阵
  labels_expanded.shape = torch.Size([8, 1])
  same_label_mask.shape = torch.Size([8, 8])
  相同标签对数量: 4 (去掉对角线，除以2去重)
  不同标签对数量: 20 (去掉对角线，除以2去重)

【核心代码步骤3】挖掘困难正样本对
  锚点0的正样本候选: [1]
  锚点1的正样本候选: [0]
  锚点2的正样本候选: [3]
  锚点3的正样本候选: [2]
  锚点4的正样本候选: [5]
  锚点5的正样本候选: [4]
  锚点6的正样本候选: [7]
  锚点7的正样本候选: [6]
  正样本对数量: 0

【核心代码步骤4】挖掘困难负样本对
  负样本锚点0的负样本候选: [2, 3, 4, 5, 6, 7]
  负样本锚点1的负样本候选: [2, 3, 4, 5, 6, 7]
  负样本锚点2的负样本候选: [0, 1, 4, 5, 6, 7]
  负样本锚点3的负样本候选: [0, 1, 4, 5, 6, 7]
  负样本锚点4的负样本候选: [0, 1, 2, 3, 6, 7]
  负样本锚点5的负样本候选: [0, 1, 2, 3, 6, 7]
  负样本锚点6的负样本候选: [0, 1, 2, 3, 4, 5]
  负样本锚点7的负样本候选: [0, 1, 2, 3, 4, 5]
  负样本对数量: 37

【结果】
  正样本对: 0个
  负样本对: 37个
  前5个正样本对

In [ ]:
import torch

descriptors = torch.tensor([[ 0.5484, -1.8406,  0.3203, -0.4526],
        [-0.4881,  0.4921, -0.2268,  0.4315],
        [ 0.7441, -1.3973,  0.4317, -1.0654],
        [-0.7276,  0.2828,  1.8962,  0.4187],
        [ 1.4946, -1.1203, -1.5092,  0.0088],
        [-1.3241,  1.1952, -0.4905,  0.4526],
        [-0.6548, -0.2782, -0.6012,  0.1352],
        [ 0.2919,  0.8895,  0.1728, -0.4657]]) # [8, 4] - 8个样本，每个4维特征
labels = torch.tensor([0, 0, 1, 1, 2, 2, 3, 3])  # 8个标签
print(f"  descriptors.shape = {descriptors.shape}")
print(f"  labels = {labels.tolist()}")
print(f"\n  descriptors 矩阵 (8个样本 × 4维特征):")
# print(f"  {descriptors}")

def cosine_similarity_matrix(x):
    """计算余弦相似度矩阵"""
    x_norm = x / (x.norm(dim=1, keepdim=True) + 1e-8)
    sim_matrix = torch.mm(x_norm, x_norm.t())
    return sim_matrix

cosine_sim = cosine_similarity_matrix(descriptors)
print(f"  余弦相似度矩阵 [8, 8]:")
# print(f"  {cosine_sim}")

# 创建标签匹配矩阵
labels_expanded = labels.unsqueeze(1)  # [8, 1]
same_label_mask = (labels_expanded == labels_expanded.t()).float()  # [8, 8]

print(f"\n  标签匹配矩阵 [8, 8] (1=相同标签, 0=不同标签):")
print(f"  {same_label_mask.int()}")

from pytorch_metric_learning import miners
from pytorch_metric_learning.distances import CosineSimilarity

miner = miners.MultiSimilarityMiner(epsilon=0.1, distance=CosineSimilarity())
miner_outputs = miner(descriptors, labels)

tensor_a, tensor_b, tensor_c, tensor_d = miner_outputs

print(f"  Miner输出:")
print(f"    tensor_a (正样本锚点): {tensor_a.tolist()}")
print(f"    tensor_b (正样本): {tensor_b.tolist()}")
print(f"    tensor_c (负样本锚点): {tensor_c.tolist()}")
print(f"    tensor_d (负样本): {tensor_d.tolist()}")

print(f"\n  挖掘出的样本对数量:")
print(f"    正样本对: {len(tensor_a)}个")
print(f"    负样本对: {len(tensor_c)}个")


# 展示前几个正样本对的详细信息
print(f"\n  正样本对详情 (前5个):")
for i in range(min(5, len(tensor_a))):
    anchor_idx = tensor_a[i].item()
    pos_idx = tensor_b[i].item()
    anchor_label = labels[anchor_idx].item()
    pos_label = labels[pos_idx].item()
    sim_val = cosine_sim[anchor_idx, pos_idx].item()
    match = "✓" if anchor_label == pos_label else "✗"
    print(f"    对{i+1}: 锚点[{anchor_idx}](标签={anchor_label}) <-> 正样本[{pos_idx}](标签={pos_label}) "
          f"相似度={sim_val:.4f} {match}")

  descriptors.shape = torch.Size([8, 4])
  labels = [0, 0, 1, 1, 2, 2, 3, 3]

  descriptors 矩阵 (8个样本 × 4维特征):
  余弦相似度矩阵 [8, 8]:

  标签匹配矩阵 [8, 8] (1=相同标签, 0=不同标签):
  tensor([[1, 1, 0, 0, 0, 0, 0, 0],
        [1, 1, 0, 0, 0, 0, 0, 0],
        [0, 0, 1, 1, 0, 0, 0, 0],
        [0, 0, 1, 1, 0, 0, 0, 0],
        [0, 0, 0, 0, 1, 1, 0, 0],
        [0, 0, 0, 0, 1, 1, 0, 0],
        [0, 0, 0, 0, 0, 0, 1, 1],
        [0, 0, 0, 0, 0, 0, 1, 1]], dtype=torch.int32)
  Miner输出:
    tensor_a (正样本锚点): [0, 1, 2, 3, 4, 5, 6, 7]
    tensor_b (正样本): [1, 0, 3, 2, 5, 4, 7, 6]
    tensor_c (负样本锚点): [0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 2, 2, 2, 3, 3, 3, 3, 4, 4, 4, 4, 4, 5, 5, 5, 5, 6, 6, 6, 6, 6, 6, 7, 7, 7, 7, 7, 7]
    tensor_d (负样本): [5, 7, 3, 6, 4, 2, 4, 7, 3, 6, 5, 7, 4, 0, 0, 7, 1, 5, 1, 7, 6, 2, 0, 3, 7, 6, 1, 3, 2, 0, 4, 1, 5, 0, 4, 2, 1, 3, 5]

  挖掘出的样本对数量:
    正样本对: 8个
    负样本对: 39个
